# Этап 5: малые нелинейные модели направления

Последний дешёвый тест: способны ли shallow HistGradientBoosting/ExtraTrees извлечь взаимодействия между `reaction_core`, типом события, surprise, сектором и рыночным режимом.

## План и замороженный gate

Primary challenger — HistGradientBoosting; ExtraTrees и фиксированный 50/50 blend — diagnostics. Baselines `reaction_core` и `structured_logistic` заблокированы из этапа 4. Evaluation: 480 событий, 7 walk-forward folds, два месяца validation, embargo 72 часа, purge по `event_group_id`. Gate: hit rate ≥58%, нижняя cluster-CI >50%, минимум 5/7 положительных folds и нижняя cluster-CI Δ ROC-AUC против `reaction_core` >0.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
experiment_dir = (
    cwd
    if (cwd / 'run_experiment.py').exists()
    else cwd / 'experiments' / 'direction_stage5'
)
repo_root = experiment_dir.parents[1]
sys.path.insert(0, str(repo_root))
from experiments.direction_stage5.run_experiment import (  # noqa: E402
    assert_feature_contract,
    run_experiment,
)
from experiments.finbert_stage1.run_experiment import sha256_file  # noqa: E402
from experiments.paths import dataset_path_from_environment, stage_artifact_directory  # noqa: E402

dataset_path = dataset_path_from_environment()
artifact_dir = stage_artifact_directory('direction_stage5')
market_path = stage_artifact_directory('market_stage2') / 'market_features.csv'
structured_path = (
    stage_artifact_directory('direction_stage4') / 'structured_features.csv'
)
print(dataset_path)

In [ ]:
# По умолчанию читаем проверенные артефакты; полный пересчёт: RUN_STAGE5=1.
if os.environ.get('RUN_STAGE5') == '1':
    result = run_experiment(
        dataset_path,
        artifact_dir,
        market_features_path=market_path,
        structured_features_path=structured_path,
    )
else:
    result = json.loads((artifact_dir / 'metrics.json').read_text())
print(result['experiment'], result['created_at'])

## Автоматический контроль

In [ ]:
predictions = pd.read_csv(artifact_dir / 'direction_predictions.csv')
folds = pd.read_csv(artifact_dir / 'fold_metrics.csv')
assert_feature_contract()
assert len(predictions) == 480 and predictions.id.nunique() == 480
assert predictions.fold.nunique() == 7
assert set(folds.model) == {
    'hist_gradient_boosting',
    'extra_trees',
    'nonlinear_blend',
}
assert len(folds) == 21
assert all(not gate['passed'] for gate in result['deployment_gates'].values())
assert result['reference_baselines']['majority_class_accuracy'] == 8 / 15
for filename, expected in result['artifact_hashes'].items():
    assert sha256_file(artifact_dir / filename) == expected
print(
    'QA passed: leakage contract, 480 unique OOF rows, 7 folds, '
    '21 selected fold-model records, hashes and failed gates.'
)

## Основные результаты

In [ ]:
rows = []
for name, metrics in result['metrics'].items():
    rows.append({
        'model': name,
        'ROC-AUC': metrics['roc_auc'],
        'hit rate': metrics['accuracy'],
        'balanced accuracy': metrics['balanced_accuracy'],
        'hit CI low': metrics['accuracy_cluster_bootstrap']['ci95_low'],
        'hit CI high': metrics['accuracy_cluster_bootstrap']['ci95_high'],
        'positive folds': metrics['positive_folds'],
    })
display(pd.DataFrame(rows).set_index('model').round(4))
print('Reference baselines:', result['reference_baselines'])

In [ ]:
rows = []
for name, comparison in result['paired_comparisons_vs_reaction_core'].items():
    rows.append({
        'model': name,
        'Δ ROC-AUC': comparison['roc_auc_delta']['estimate'],
        'AUC Δ CI low': comparison['roc_auc_delta']['ci95_low'],
        'AUC Δ CI high': comparison['roc_auc_delta']['ci95_high'],
        'Δ hit rate': comparison['accuracy_delta']['estimate'],
        'hit Δ CI low': comparison['accuracy_delta']['ci95_low'],
        'hit Δ CI high': comparison['accuracy_delta']['ci95_high'],
    })
display(pd.DataFrame(rows).set_index('model').round(4))
display(pd.DataFrame(result['deployment_gates']).T)

## Это не просто majority prediction

In [ ]:
prediction_rates = []
for name in result['metrics']:
    prediction_rates.append({
        'model': name,
        'predicted up rate': predictions[f'prediction_{name}'].mean(),
        'hit rate': result['metrics'][name]['accuracy'],
        'balanced accuracy': result['metrics'][name]['balanced_accuracy'],
    })
display(pd.DataFrame(prediction_rates).set_index('model').round(4))
print(
    'Blend predicts both classes, but its 53.33% hit rate merely equals '
    'the accuracy of always-down on this population.'
)

## Устойчивость и стоимость

In [ ]:
fold_accuracy = pd.DataFrame({
    name: metrics['fold_accuracy']
    for name, metrics in result['metrics'].items()
})
display(fold_accuracy.round(4))
display(pd.DataFrame(result['resources']).T.round(2))
display(
    folds[[
        'fold',
        'model',
        'validation_roc_auc',
        'test_roc_auc',
        'test_accuracy',
        'serialized_size_bytes',
        'parameters',
    ]]
)

## Config 6 и итоговый график

In [ ]:
config_rows = []
for name, metrics in result['config6_population']['models'].items():
    config_rows.append({
        'model': name,
        'hit rate': metrics['hit_rate'],
        'Δ vs config 6': metrics['delta_vs_config_6']['estimate'],
        'CI low': metrics['delta_vs_config_6']['ci95_low'],
        'CI high': metrics['delta_vs_config_6']['ci95_high'],
    })
print('config 6 hit rate:', result['config6_population']['config_6_hit_rate'])
display(pd.DataFrame(config_rows).set_index('model').round(4))
display(Image(filename=str(artifact_dir / 'model_comparison.png')))

## Итог

Ни одна малая nonlinear-модель не прошла gate. ExtraTrees поднял наблюдаемый ROC-AUC до 0,5369, но 95% CI эффекта включает ноль и результат выше 50% только в 2/7 folds. Blend имеет 53,33% hit rate, равный majority baseline, при CI [48,95%; 57,71%]. После пяти отрицательных directional-этапов дальнейший model tuning на текущем датасете следует остановить: direction убрать из product promise, а сервис строить вокруг доказанно работающей значимости и объяснения событий.